In [0]:
import numpy as np
import pandas as pd
import plotly.express as px
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
# DESCRIBE DETAIL helps to monitor dim_carburant freshness
brze_dim_carburant = spark.sql("DESCRIBE DETAIL kyc.bronze.brze_dim_carburant ")
#  brze_dim_geo freshness
brze_dim_geo       = spark.sql("DESCRIBE DETAIL kyc.bronze.brze_dim_geo")
# brze_dim_service freshness
brze_dim_service   = spark.sql("DESCRIBE DETAIL kyc.bronze.brze_dim_service")
# brze_dim_station freshness
brze_dim_station   = spark.sql("DESCRIBE DETAIL kyc.bronze.brze_dim_station")
# brze_fait_prix freshness
brze_fait_prix   = spark.sql("  DESCRIBE DETAIL kyc.bronze.brze_fait_prix")
# brze_fait_rupture freshness
brze_fait_rupture   = spark.sql("DESCRIBE DETAIL kyc.bronze.brze_fait_rupture")

# display(brze_dim_carburant)

display(brze_dim_carburant)

In [0]:
# lastModified time gives us a rudimentary view into dim_carburant_freshness

dim_carburant_update_times = brze_dim_carburant.select(
    "id",
    "name",
    "lastModified"
    )\
    .withColumn("measureAt", F.lit(datetime.now().isoformat()))

In [0]:

# From here, we have appended to a freshness-tracking table called bronze_update_times
dim_carburant_update_times.write.mode("append").saveAsTable("kyc.bronze.bronze_update_times")

In [0]:
time_series = spark.sql("""                  
            SELECT DISTINCT 
            measureAt,
            lastModified,
            1 AS val
            FROM kyc.bronze.bronze_update_times
            ORDER BY measureAt ASC
            LIMIT 400
    """).toPandas()
time_series.lastModified = pd.to_datetime(time_series.lastModified)
time_series.measureAt = pd.to_datetime(time_series.measureAt)

In [0]:
px.scatter(x =time_series.lastModified, y = time_series.measureAt)

In [0]:
# Use some pandas ingenuity to display the delay times as a bar chart
differences = time_series.assign(delaySeconds = lambda x: (x.lastModified - time_series.shift(periods=1, axis="rows").lastModified)/ np.timedelta64(1, 's'))
differences = differences.loc[~(differences.delaySeconds == 0)].dropna().reset_index()
px.bar(x = differences.lastModified, y = differences.delaySeconds)